In [ ]:
import math
import numpy as np
import copy

FRAMES = 20
POPULATION_SIZE = 10
MAX_SPEED = 10

In [ ]:
class NeuralNetwork:
    def __init__(self, p_pos, f_pos):
        self.player_pos = p_pos
        self.fruit_pos = f_pos
        self.layer_1_weights = self.generate_random_weights(3, 4)
        self.layer_1_bias = self.generate_random_biases(3)
        self.layer_2_weights = self.generate_random_weights(2, 3)
        self.layer_2_bias = self.generate_random_biases(2)

    def generate_random_weights(self, rows, cols):
        return np.random.uniform(-1, 1, (rows, cols))

    def generate_random_biases(self, size):
        return np.random.uniform(-1, 1, size)

    def forward(self):

        raw_values = [self.player_pos[0], self.player_pos[1], self.fruit_pos[0], self.fruit_pos[1]]

        # LAYER 1
        Z_layer_1 = []
        for neuron_weights in self.layer_1_weights:  # for each row of weights
            z_in_row = sum(n * w for n, w in zip(raw_values, neuron_weights))  # multiply and sum the values and weights
            Z_layer_1.append(z_in_row)

        Z_layer_1 = [v + b for v, b in zip(Z_layer_1, self.layer_1_bias)]  # add bias

        A_layer_1 = [math.tanh(v) for v in Z_layer_1]  # normalize

        # LAYER 2
        Z_layer_2 = []
        for neuron_weights in self.layer_2_weights:  # for each row of weights
            Z_out_row = sum(n * w for n, w in zip(A_layer_1, neuron_weights))  # multiply and sum the values and weights
            Z_layer_2.append(Z_out_row)

        Z_layer_2 = [v + b for v, b in zip(Z_layer_2, self.layer_2_bias)]  # add bias

        A_layer_2 = [math.tanh(v) for v in Z_layer_2]  # normalize

        # OUTPUT
        angle = (A_layer_2[0] + 1) / 2 * 360
        speed = (A_layer_2[1] + 1) / 2 * MAX_SPEED
        
        angle_rad = math.radians(angle)
        self.player_pos[0] += math.cos(angle_rad) * speed
        self.player_pos[1] += math.sin(angle_rad) * speed

        return {"angle": angle, "speed": speed}

    def mutate(self, mutation_rate=0.01):
        self.layer_1_weights += np.random.uniform(-mutation_rate, mutation_rate, self.layer_1_weights.shape)
        self.layer_1_bias += np.random.uniform(-mutation_rate, mutation_rate, self.layer_1_bias.shape)
        self.layer_2_weights += np.random.uniform(-mutation_rate, mutation_rate, self.layer_2_weights.shape)
        self.layer_2_bias += np.random.uniform(-mutation_rate, mutation_rate, self.layer_2_bias.shape)

In [2673]:

neural_networks = [NeuralNetwork([2, 3], [80, 90]) for _ in range(POPULATION_SIZE)]

We run each generation for 20 frames, then we get the one that got the closest to the fruit, deep copy it 10 times, so the new generation has the previous best weights, and mutate a bit to add some change.

In [ ]:
for g in range(FRAMES):
    for n in neural_networks:
        n.forward()

closest = 1000
best_ai = FRAMES * MAX_SPEED * 2

for n in neural_networks:
    dx = n.player_pos[0] - n.fruit_pos[0]
    dy = n.player_pos[1] - n.fruit_pos[1]
    distance = math.sqrt(dx**2 + dy**2)
    if distance < closest:
        closest = distance
        best_ai = n
    # print(n.player_pos, distance)


print(best_ai.player_pos, closest)
best_ai.player_pos = [2, 3]

neural_networks = [copy.deepcopy(best_ai) for _ in range(POPULATION_SIZE)]

for n in neural_networks[1:]:
    n.mutate()

[80.00026472902935, 89.9881598211564] 0.011843137950196704
